### PINN (Physics Informed Neural Network)

1. Solving A 1D problem using PINN



# 1. Problem Formulation

Here we want to solve a basic *ODE* with use of PINN and we will use normal iterative based methods/normal PDE solve to genrate the ground truth. 

The given PDE to solve is:

$\frac{\partial u}{\partial x} = u$

## 2. Build a Model

Build a PINN Neural network model that can add layers and neurons as per required.

- generalized enough that can add take input neurons, number of hidden layers, and output neurons 
- nn.Tanh() as activation that puts everything to good shape
- No dropout
- Fully cconneccted layers

In [1]:
!pip install torch

In [2]:
# Import important libraries
import torch
import torch.nn as nn 

import numpy as np
import matplotlib.pyplot as plt

In [3]:
# device agnostic code

def add_device() -> torch.device:
    "checck if cuda is available or not. If available set device to cuda else cpu"
    if torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    
    print(f"Using device: {device}")

    return device

In [4]:
device = add_device()

Using device: cuda


In [6]:
# Neural Network

class PINNModel(nn.Module):

    def __init__(self, 
                 in_channels: int,
                 hidden_layers: int,
                 out_channels:int) -> None:
        
        self.fc1 = nn.Sequential(
            nn.Linear(in_features=in_channels, out_features=hidden_layers),
            nn.Tanh(),
            nn.Linear(in_features=hidden_layers, out_features=hidden_layers),
            nn.Tanh(),
            nn.Linear(in_features=hidden_layers, out_features=hidden_layers),
            nn.Tanh(),
            nn.Linear(in_features=hidden_layers, out_features=out_channels),
            nn.Tanh(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc1(x)

In [ ]:
from typing import List

## Training step
def training_step(X_train: torch.Tensor,
                  y_train: torch.Tensor,
                  model: nn.Module,
                  loss_fn: nn.Module,
                  optimizer: torch.optim.Optimizer,
                  ) -> torch.Tensor:
    
    ## 0. Put model to train mode
    model.train()

    # 1. Forward pass
    y_pred = model(X_train)

    # 2. Loss function (Residual) computation
    loss_pde = loss_fn(y_train, y_pred)
    ## loss of physics
    loss_phy = loss_physics()

    # 3. Optimizer zero grad
    optimizer.zero_grad()

    # 4. Backpropagation
    loss.backward()

    # 5. Optimizer step
    optimizer.step()

    return loss

In [10]:
## Validation step

def validation_step(X_val: torch.Tensor,
                    y_val: torch.Tensor,
                    model: nn.Module,
                    loss_fn: nn.Module,
                    ) -> torch.Tensor:
    
    ## 0. Put model to validation mode
    model.eval()

    with torch.inference_mode():

        # 1. Forward pass
        y_pred = model(X_val)

        # 2. Loss function (Residual) computation
        loss = loss_fn(y_val, y_pred)

    return loss

In [12]:
## physics loss computation
def loss_phy():
    pass